# Workspace for Fourier Tasks

- By: (redacted), 2025
- Corresponding paper (redacted), 2026
- Licence: MIT
- [Click here for the Fourier source repository (redacted)](https://fake.url)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio

## Experimentation

Experiment with the code below for creating signals and
using ipython's `Audio` functionality to hear them back.

Try altering the values of the sample rate (`sr`), seconds (`T`), fundamental frequency (`hz`), and amplitude (`a`).

In [ ]:
sr = 22050 # sample rate
T = 1.0    # seconds
hz = 440   # fundamental frequency
a = 1    # amplitude
time_array = np.linspace(0, T, int(T * sr))

In [ ]:
x_1 = a * np.sin(2 * np.pi * hz * time_array)

Let's explore this data ...
... as audio ...

In [ ]:
Audio(x_1, rate=sr)

... and also visually ...

In [ ]:
plt.plot(x_1[:500])
plt.show()

Now let's combine more than one signal.

In [ ]:
hz = 800
x_2 = a * np.sin(2 * np.pi * hz * time_array)

In [ ]:
Audio(x_2, rate=sr)

In [ ]:
Audio(x_1 + x_2, rate=sr)

In [ ]:
plt.plot(x_1[:500] + x_2[:500])
plt.show()

# Task and Reference

## Background

Simple sounds are periodic.
The simplest consist of a single periodic sine wave.
This is a synthetic sound: nature is almost more complex.
A first step towards more realistic sounds involves
combining one frequency – the 'fundamental' – with 'partials':
other periodic signals with frequencies at integer multiples of the fundamental.

## Task

- Type: Implement roundtrip.
- Task: Generalise the creation of composite signals as follows:
    - Chose a fundamental frequency e.g., 100Hz
    - Create a periodic function with that frequency (sin or cos).
    - Create further periodic functions with integer multiples of this frequency (n) and lower amplitudes, e.g.:
      - for every integer multiple (2, 3, 4, ...)
      - the amplitude is reduced (1/2, 1/3, 1/4, ... )
    - Combine them to make a pseudo-realistic composite waveform.
      - Hint: `Audio({time_array_1} + {time_array_1}, rate=sr)`
    - Plot the composite signal and all components (_time_ against amplitude)
    - Apply DFT to this function
      - Bonus: Implement DFT from scratch.
    - Plot this DFT (_frequency_ against amplitude)
    - Verify that the DFT returns the same frequency:amplitude pairs that you input.
- Reference implementation: `composite_signal.run()` (and for the bonus DFT from scratch see the `fourier` task).


## Workspace

## Reference

In [ ]:
from typing import Union

def simple_sin(
        a: Union[int, float],
        hz: Union[int, float],
        t: Union[int, float, np.array]
):
    """
    :param a: amplitude
    :param hz: frequency
    :param t: time (point or array)
    """
    return a * np.sin(2 * np.pi * hz * t)

In [ ]:
s = simple_sin(1, 100, 0.0325)
np.isclose(s, 1)

In [ ]:
def fundamental_w_partials(
        fundamental: int = 100,
        multiples=None,
) -> np.array:
    """
    Support function for creating a composite signal with
    a fundamental frequency
    and several overtones with
    each frequency as an integer multiple of the fundamental and
    each max amplitude by inverse proportion.

    :param fundamental: the fundamental frequency in Hertz
    :param multiples: a list of integer multiples to create overtones.
    :return: dict with each overtone (the multiples given) and 1 / overtone as a max amplitude.
    """

    overtone_magnitude_dict = {fundamental: 1}

    if multiples is None:
        multiples = [1, 2, 4]

    for m in multiples:
        overtone_magnitude_dict[m * fundamental] = 1 / m

    return overtone_magnitude_dict

In [ ]:
fundamental_w_partials(100, [1, 2, 4])

In [ ]:
def composite_signal(
        t: Union[int, float, np.array],
        overtone_magnitude_dict: dict = fundamental_w_partials()
) -> float:
    """
    Create a composite signal from a time array,
    and a set of frequency-magnitude pairs for simple sine waves.

    :param t: time point or array
    :param overtone_magnitude_dict: frequency-magnitude pairs. See `fundamental_w_partials` for an example.
    """
    out_val = 0
    for hz in overtone_magnitude_dict:
        out_val += simple_sin(overtone_magnitude_dict[hz], hz, t)
    return out_val

In [ ]:
def plot_signal(
        time_array: np.array,
        plot_proportion: float = 0.1,
        overtone_magnitude_dict: Union[dict, None] = None,
        plot_components: bool = True,
) -> None:
    """
    Plot a `time_array` against both a composite signal and the components that make it up.

    :param time_array: a numpy array of time increments
    :param plot_proportion: how much of the sample to plot. Float, default 0.1 (i.e., 10%).
    :param overtone_magnitude_dict: see `fundamental_w_partials` above
    :param plot_components: if True, Create a second subplot with the signal and components shown separately.
    :return: None (plot)
    """
    plt.figure(figsize=(10, 6))
    axis_label_size, legend_label_size = 14, 13
    plt.subplot(2, 1, 1)  # position above/below

    if 0 < plot_proportion <= 1:
        plot_range = int(len(time_array) * plot_proportion)
    else:
        raise ValueError("The `plot_proportion` must be greater than 0 and less than or equal to 1.")

    # Prep and plot composite
    if not overtone_magnitude_dict:
        overtone_magnitude_dict = fundamental_w_partials()
    y = composite_signal(time_array, overtone_magnitude_dict=overtone_magnitude_dict)
    plt.plot(time_array[:plot_range], y[:plot_range], label='signal')

    plt.ylabel('Amplitude', fontsize=axis_label_size)
    plt.axis('tight')
    plt.grid()
    plt.xlim(time_array[0], time_array[plot_range])
    plt.legend(loc='upper right', shadow=False, fontsize=legend_label_size)

    if plot_components:
        plt.subplot(2, 1, 2)  # position above/below
        plt.plot(time_array, y, label='signal')  # as before, signal
        count = 1
        for hz in overtone_magnitude_dict:  # now the reference/components
            y = simple_sin(overtone_magnitude_dict[hz], hz, time_array)
            plt.plot(time_array[:plot_range], y[:plot_range], "--", label=f'component {count}')
            count += 1
        plt.ylabel('Amplitude', fontsize=axis_label_size)
        plt.axis('tight')
        plt.grid()
        plt.xlim(time_array[0], time_array[plot_range])

    plt.xlabel('Time (seconds)', fontsize=axis_label_size)
    plt.legend(loc='upper right', shadow=False, fontsize=legend_label_size)
    plt.tight_layout()

    plt.show()

In [ ]:
plot_signal(time_array, plot_proportion=0.1, overtone_magnitude_dict=fundamental_w_partials())

In [ ]:
def fft_from_composite(
        composite: Union[np.array, None] = None,
        plot_proportion: float = 0.5,
) -> None:
    """
    Calculate and plot the Fourier transform.

    :param composite: The signal. If none, then it creates a demo from hard-coded values (1, 2, 4).
    :param plot_proportion: a float, greater than 0 and leq 1. E.g.,
        if 1 then return full results (at least after the abs),
        if 0.5 then return half removing the symmetry,
        if outside the range 0-1, then raise ValueError.
    :return: None (plot)
    """
    if not composite:
        fp = fundamental_w_partials(fundamental=100, multiples=[1, 2, 4])
        composite = composite_signal(time_array, overtone_magnitude_dict=fp)

    X = np.fft.fft(composite)

    X_mag = np.absolute(X)  # NB: len(X_mag) still = sr = 22050

    f = np.arange(0, sr, 1)  # 0 to 22050, step size 1. So [0, 1, 2, ... 22050]. Or equivalently:
    # f = np.linspace(0, sr, len(X_mag))  # 0 to 22050 with 22050 increments

    plt.figure(figsize=(10, 6))

    if 0 < plot_proportion <= 1:
        plt.plot(f[:int(len(f) * plot_proportion)], X_mag[:int(len(X_mag) * plot_proportion)])
    else:
        raise ValueError("The `plot_proportion` must be Greater than nought and less than or equal to 1.")

    axis_label_size = 14
    plt.ylabel('FFT Amplitude $|X(freq)|$', fontsize=axis_label_size)
    plt.xlabel('Frequency, $Hz$', fontsize=axis_label_size)

    plt.grid()

    plt.show()

In [ ]:
fft_from_composite(plot_proportion=0.15)